In [4]:
from ax.service.ax_client import AxClient
import sys
sys.path.append('../')
import helper_functions as hf


In [6]:
iteration_to_update = 3
optimizer_file_path = 'iteration_' + str(iteration_to_update) + '/optimizer/optimizer_'
ax_to_update_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"



In [7]:
ax_to_update = AxClient.load_from_json_file(ax_to_update_path)
trials_to_update = ax_to_update.get_trials_data_frame()
trials_to_update


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,surf_3_conc,drug_conc,surf_1,surf_2,surf_3
0,0,0_0,COMPLETED,GenerationStep_0,50.000000,0.2063,0.3073,0.0373,6.342538,1.979255e+01,10.062115,25.0,s8,s6,s7
1,1,1_0,COMPLETED,GenerationStep_0,47.663097,0.2063,0.3073,0.0373,25.388954,2.403562e+00,19.870581,25.0,s3,s2,s6
2,2,2_0,COMPLETED,GenerationStep_0,50.000000,0.4045,0.4196,0.0728,0.561928,2.777170e+01,0.445989,25.0,s5,s7,s3
3,3,3_0,COMPLETED,GenerationStep_0,24.577678,0.4045,0.4196,0.0728,14.076332,3.422734e+00,7.078612,25.0,s3,s3,s8
4,4,4_0,COMPLETED,GenerationStep_0,50.000000,0.2962,0.4364,0.0493,13.456925,3.146619e+01,1.570027,25.0,s1,s4,s5
5,5,5_0,COMPLETED,GenerationStep_0,50.000000,0.2962,0.4364,0.0493,20.019197,1.454911e+01,11.283737,25.0,s6,s8,s7
6,6,6_0,COMPLETED,GenerationStep_0,50.000000,0.3528,0.2810,0.0711,2.750314,1.224179e+01,9.081532,25.0,s1,s1,s6
7,7,7_0,COMPLETED,GenerationStep_0,23.780974,0.3528,0.2810,0.0711,5.021025,8.626882e-01,17.897261,25.0,s6,s5,s1
8,8,8_0,COMPLETED,GenerationStep_1,50.000000,0.2063,0.3073,0.0373,5.668198,6.320464e+00,11.327995,25.0,s6,s5,s1
9,9,9_0,COMPLETED,GenerationStep_1,50.000000,0.2063,0.3073,0.0373,5.247319,6.494840e+00,11.959137,25.0,s6,s5,s1


In [8]:
def update_data_to_optimizer(ax_client, list_of_new_failures):

    # Load the existing optimizer state
    before_update_path = optimizer_file_path + f"{iteration_to_update}_before_update.json"
    updated_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

    ax_client.save_to_json_file(before_update_path)

    # Fetch current trials
    trials_df = ax_client.get_trials_data_frame()

    for trial_index in list_of_new_failures:
        # Make sure we actually have this trial
        if trial_index not in trials_df["trial_index"].values:
            print(f"Trial {trial_index} not found – skipping.")
            continue

        # Build the forced-failure payload
        new_data = {
            "obj_surf_conc": hf.surfactant_stock_conc,
        }

        # Update the trial in-place
        ax_client.update_trial_data(trial_index=trial_index, raw_data=new_data)

    ax_client.save_to_json_file(updated_path)

    print(f"Updated trial {trial_index}: set success=0, surfactant_input=1, complexity=1")

    return ax_client

In [9]:
list_of_new_failures = [3,7]

In [10]:
print("Please double check the results you are updating before continue...")
print("*" * 100)
print("*" * 100)

df = trials_to_update[trials_to_update["trial_index"].isin(list_of_new_failures)]
df


Please double check the results you are updating before continue...
****************************************************************************************************
****************************************************************************************************


,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,surf_3_conc,drug_conc,surf_1,surf_2,surf_3
3,3,3_0,COMPLETED,GenerationStep_0,24.577678,0.4045,0.4196,0.0728,14.076332,3.422734,7.078612,25.0,s3,s3,s8
7,7,7_0,COMPLETED,GenerationStep_0,23.780974,0.3528,0.2810,0.0711,5.021025,0.862688,17.897261,25.0,s6,s5,s1


In [11]:
new_ax_client = update_data_to_optimizer(ax_to_update, list_of_new_failures)

[INFO 07-17 09:26:01] ax.service.ax_client: Added data: {'obj_surf_conc': (50.0, None)} to trial 3.
[INFO 07-17 09:26:01] ax.service.ax_client: Added data: {'obj_surf_conc': (50.0, None)} to trial 7.


Updated trial 7: set success=0, surfactant_input=1, complexity=1


In [12]:
updated_trials = new_ax_client.get_trials_data_frame()
updated_trials[updated_trials["trial_index"].isin(list_of_new_failures)]

,trial_index,arm_name,trial_status,generation_node,obj_surf_conc,Drug_MW,Drug_LogP,Drug_TPSA,surf_1_conc,surf_2_conc,surf_3_conc,drug_conc,surf_1,surf_2,surf_3
3,3,3_0,COMPLETED,GenerationStep_0,50.0,0.4045,0.4196,0.0728,14.076332,3.422734,7.078612,25.0,s3,s3,s8
7,7,7_0,COMPLETED,GenerationStep_0,50.0,0.3528,0.2810,0.0711,5.021025,0.862688,17.897261,25.0,s6,s5,s1
